# Improvement Methods Report

This notebook documents the engineering improvements applied across the stochastic TSP project.

The goal of the refactor was not to invent new algorithms from scratch, but to make the existing algorithms more robust, depot-aware, reproducible, and easier to compare fairly across random and built-in instances.

## 1. Project-wide improvements

### 1.1 Depot handling
- Added support for `depot_city=None` in the environment.
- If no depot is specified, the env now selects a random depot on construction.
- On later resets, the depot advances cyclically instead of staying hardcoded at city 1 or city 0.
- Fixed depot-specific logic so it works for all algorithms, not just one chosen baseline.

### 1.2 Action indexing and route mapping
- Removed assumptions that action `a` always means city `a + 1`.
- Added depot-relative city/action mapping helpers in the environment.
- Updated route reconstruction so every algorithm can report the correct route for any depot.

### 1.3 Reproducibility and training behavior
- Kept environment objects persistent across episodes for efficiency.
- `reset(seed=...)` is now the episode boundary for reseeding and depot advancement.
- Added explicit route and depot reporting in training/evaluation outputs.

### 1.4 Validation
- Ran syntax validation on all touched modules.
- Ensured experiment scripts still produce comparable summaries and records.

## 2. Exact deterministic TSP improvements (`classic_DP.py`)

### Improvements made
- Added a `depot_city` parameter.
- Re-indexed the distance matrix internally so the Held-Karp DP can start from any depot.
- Restored the final route back to original city IDs before returning results.
- Preserved the existing optimal-subtour structure while making the solver depot-agnostic.

### Why this matters
- The deterministic baseline can now be compared fairly against stochastic RL policies when the start city is not fixed.
- This avoids silently penalizing methods that were previously forced to start at city 1 or city 0.

## 3. Exact stochastic TSP improvements (`value_iteration.py` and `exact_safety.py`)

### Value iteration
- Uses the env’s current depot city instead of assuming city 0.
- Builds state space and route reconstruction consistently around the active depot.
- Converts actions back into city IDs via the env mapping.

### Exact safety wrapper
- Propagates depot-aware DP and VI execution through the guarded subprocess runner.
- Keeps the memory/timeout safety wrapper intact while making outputs consistent with the current depot.

### Why this matters
- Exact methods now match the same depot policy used by the environment and learning agents.
- This prevents mismatched evaluation routes when the depot changes across runs.

## 4. Heuristic improvements (`heuristic.py`)

### Improvements made
- Fixed the heuristic to always use depot city 1.
- Prevented the heuristic from inheriting random/cyclic depot behavior.
- Updated route reporting to use the env’s actual depot and action mapping.

### Why this matters
- The heuristic is meant to serve as a stable, interpretable baseline.
- Pinning it to a fixed depot makes its results reproducible and easier to interpret across experiments.

### Practical effect
- The heuristic no longer changes starting city between episodes or runs.
- It stays comparable as a fixed-policy baseline while the other methods use the depot policy introduced in the environment.

## 5. Policy-based RL improvements (`RL_policy_based.py`)

### Improvements made
- Made episode rollouts depot-aware.
- Removed hardcoded `action + 1` logic from route reconstruction.
- Stored full episode trajectories explicitly as lists of states, actions, masks, next states, and dones.
- Added `action_cities` handling for depot-relative action orders.
- Added robust critic/optimizer narrowing to keep PPO and A2C type-safe.

### Why this matters
- RL policies can now train and evaluate correctly even when the depot changes.
- The update code no longer assumes a fixed city order.
- The agent’s logged route matches the actual environment behavior.

### Training behavior
- REINFORCE, AC, and A2C remain clean on-policy updates.
- PPO keeps its own clipped objective, GAE, and mini-batch epochs.
- All methods now see the depot-aware environment interface consistently.

## 6. Pointer Network improvements (`RL_pointer_network.py`)

### Improvements made
- Made the actor accept depot-relative action city ordering.
- Updated the pointer attention path so it no longer assumes non-depot cities are fixed at `1..n-1`.
- Kept the encoder-decoder attention structure intact while making the policy compatible with arbitrary depots.
- Added consistent route tracking with the actual depot city.

### Architecture summary
- City encoder: linear projection + ReLU over full distance rows.
- Decoder: LSTMCell.
- Glimpses: attention refinement layers.
- Pointer head: masked attention over candidate cities.
- Critic: city encoder + pooled embedding + 4-dim observation.

### Why this matters
- The pointer network is now a genuine drop-in RL policy for the depot-aware environment.
- It can be used in fair experiments with arbitrary depots and random instances.

## 7. Training loop improvements (`RL_trainer.py`)

### Improvements made
- Reused one env object per worker rather than creating a new env each episode.
- Called `reset(seed=...)` at episode boundaries to reseed and advance the depot.
- Passed per-env `action_cities` into batched policy evaluation.
- Stored routes starting from each env’s actual depot city.
- Reconstructed final episode routes using the env’s mapping instead of fixed city offsets.

### Why this matters
- Training is more efficient and less object-heavy.
- Parallel worker behavior is reproducible and consistent with the depot cycle.
- RL statistics now correspond to the actual environment, not a hardcoded start node.

## 8. Experiment orchestration improvements (`EXP_ALL.py`)

### Improvements made
- Added a planning env so the exact DP baseline can use the same depot as the seeded evaluation env.
- Kept random-instance generation and built-in instance selection unchanged, but made evaluation routes depot-aware.
- Preserved comparison tables and CSV outputs while making them compatible with the new depot policy.
- Fixed random-matrix printing type issues for optional matrices.

### Why this matters
- Fair comparison requires all methods to agree on the start city.
- This change ensures the baseline and learned methods are evaluated under the same depot policy.

## 9. Summary of algorithmic engineering tricks used

Across the project, the following practical engineering improvements were applied:

1. **Depot-aware abstractions**
   - Replace hardcoded city 1 / city 0 assumptions with a proper depot model.

2. **State and route canonicalization**
   - Convert internal depot-relative action orders back to original city IDs for reporting.

3. **Persistent worker environments**
   - Reuse env objects during training for efficiency, while reseeding/resetting per episode.

4. **Fair evaluation alignment**
   - Make exact, RL, pointer, and heuristic methods comparable on the same instance/depot setup.

5. **Safety wrappers**
   - Keep exact methods behind memory and time guards so large instances do not crash the run.

6. **Type and interface hardening**
   - Fix route collection and optimizer/critic narrowing issues that could break under static analysis.

7. **Preserved baseline clarity**
   - Keep the heuristic fixed to depot 1 so it remains a stable baseline rather than a moving target.

## 10. Notes and limitations

- This notebook reports improvements that are realistic and already integrated into the current project.
- It does not claim to exhaust every possible TSP optimization technique from the literature.
- Some advanced tricks, such as beam search decoding, stronger pruning in exact solvers, curriculum training schedules, or more complex graph neural networks, are possible future work.
- The present refactor focused on correctness, fair comparison, and depot-aware reproducibility first.